### Objective
This notebook uses IMDB reviews datasets to finetune distil-BERT. Inspired by knowledge article from Hugging face

### Step 1: Install dependencies

In [1]:
import transformers
transformers.logging.set_verbosity_error()

In [2]:
# login into hugging face
from huggingface_hub import notebook_login
notebook_login()

### Step 2: Load IMDB dataset

In [3]:
from datasets import load_dataset
imdb = load_dataset("imdb")

In [4]:
imdb["test"][0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

### Step 3: Preprocess the data
Use distilBERT tokenizer

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [6]:
# preprocess function to tokenize the text; truncate the text if it is longer than dister BERT maximum input length
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True)

In [7]:
# Use map function from datasets package to apply the preprocess function on the entire dataset
tokenized_imdb = imdb.map(preprocess_function, batched=True) #batched=True helps to process multiple rows at once

In [8]:
# Create a batch of examples using DataCollatorPadding
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Step 3: Evaluate the model

In [9]:
# Include a metric during training. helps in evaluating model performance. 
# Use 'Evaluate' library to load the metric. 
import evaluate
accuracy = evaluate.load("accuracy")

In [10]:
# function to pass the predictions and labels to calculate accuracy
import numpy as np
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

### Step 4: Train

In [11]:
# Before training, create a map of expected ids to their corresponding labels
id2label = {0:"negative", 1:"positive"}
label2id = {"negative":0, "positive":1}

In [12]:
# fine tuning the model with Trainer
# load distilBERT with AutoModelForSequenceClassification
# along with expected number of labels and their mappings
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)

In [ ]:
# 3 steps to go
# 1. Define training hyper parameters in 'TrainingArguments'. The only required parameter is output_dir, which tells where to save the model. 
        # model can be pushed to huggingface hub using push_to_hub=True (need to be signed in)
        # At the end of each epoch, Trainer will evaluate the accuracy and save the model checkpoint
# 2. Pass the training arguments to Trainer along with the model, dataset, tokenizer, data collator, and compute_metrics function
# 3. Call train() method to fine tune the model 



In [13]:
# Define training hyper parameters in 'Training Arguments'
training_args = TrainingArguments(
    output_dir="distilbert-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)

In [14]:
# pass the training arguments to Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_imdb["train"],
    eval_dataset = tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator = data_collator,
    compute_metrics=compute_metrics,
)

In [15]:
import torch
import gc

gc.collect()
torch.mps.empty_cache()



In [16]:
# train the model
trainer.train()

{'loss': 0.3218, 'grad_norm': 12.599193572998047, 'learning_rate': 1.6801023672424827e-05, 'epoch': 0.3198976327575176}
{'loss': 0.2444, 'grad_norm': 8.125972747802734, 'learning_rate': 1.3602047344849649e-05, 'epoch': 0.6397952655150352}


RuntimeError: MPS backend out of memory (MPS allocated: 20.02 GB, other allocations: 7.06 GB, max allowed: 27.20 GB). Tried to allocate 192.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).